# QLoRA Stance Classification — Qwen2.5-0.5B-Instruct (Kaggle GPU, T4/P100)

Dataset: `community-datasets/catalonia_independence` (spanish config) — 3-class
stance (AGAINST / FAVOR / NEUTRAL, relabeled here to AGAINST / SUPPORT / NONE),
chosen over the original SemEval-2016 pick because it is more balanced and is a
directly loadable HF dataset with its own train/validation/test splits.

**Why GPU, not TPU:** `bitsandbytes` (the 4-bit quantization backend QLoRA
depends on) has no TPU/XLA support — only CUDA, ROCm, Intel GPU/Gaudi, and CPU.
QLoRA specifically needs that quantization kernel, so this notebook targets
Kaggle's T4/P100 GPU accelerator, not TPU.

**Rank:** fixed at 16 — the winner of an earlier 8/16/32/64 sweep on this same
setup (val Macro-F1 0.8024, converged stably through epoch 8). Rank 64 was
worse and overfit fast (early-stopped at epoch 2, Macro-F1 0.7324), so the
sweep loop below is left in place structurally but only runs once.

**What QLoRA changes vs. plain LoRA:**
- Base model loads in 4-bit NF4 (`bitsandbytes`), compute dtype bfloat16 —
  quantized storage, bf16 math. That combination *is* the mixed-precision
  requirement here; no separate `autocast`/`GradScaler` is needed.
- `prepare_model_for_kbit_training()` runs before LoRA is attached — it
  stabilizes the frozen 4-bit backbone and enables gradient checkpointing.
- The quantized model must be loaded straight onto the GPU via `device_map`;
  calling `.to("cuda")` on a 4-bit model afterward raises an error.

Run cells top to bottom. Select **GPU T4 x2** or **P100** as the accelerator
in Kaggle's notebook settings (Internet must be on).


In [9]:
# Cell 1 — Dependencies
# bitsandbytes ships on most Kaggle GPU images already; this only installs
# it if missing, so no unnecessary reinstall on a normal Kaggle session.
import importlib, subprocess, sys

def ensure(pkg, pip_name=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or pkg],
                        check=True)

ensure("bitsandbytes")
print("bitsandbytes ready")


bitsandbytes ready


In [10]:
# Cell 2 — Imports
import os, gc, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from tqdm.auto import tqdm


In [11]:
# Cell 3 — Config
SEED            = 42
MODEL_NAME      = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME    = "community-datasets/catalonia_independence"
DATASET_CONFIG  = "spanish"          # also available: "catalan"
LABELS          = ["SUPPORT", "AGAINST", "NONE"]       # canonical order used everywhere
RAW_ID2CANON    = {0: "AGAINST", 1: "SUPPORT", 2: "NONE"}  # raw LABEL ids -> canonical name
LABEL2ID        = {name: i for i, name in enumerate(LABELS)}

MAX_LEN         = 96
BATCH_SIZE      = 16
EPOCHS          = 6                  # early stopping still applies; this just caps wasted GPU time
PATIENCE        = 2                  # early-stopping patience, in epochs
LR              = 1e-4
WEIGHT_DECAY    = 0.05
WARMUP_RATIO    = 0.06
RANKS           = [16]               # fixed to rank 16: winner of the prior 8/16/32/64 sweep
                                      # (val Macro-F1 0.8024, stable through epoch 8; rank 64
                                      # collapsed and early-stopped at epoch 2 with 0.7324 —
                                      # too many trainable params for this dataset size)
LORA_ALPHA_MULT = 2                  # alpha = r * LORA_ALPHA_MULT
LORA_DROPOUT    = 0.1
LABEL_SMOOTHING = 0.1
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"]

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

assert torch.cuda.is_available(), "No GPU detected — set the Kaggle accelerator to GPU T4 x2 or P100."
DEVICE = torch.device("cuda")
print(f"Device: {DEVICE} | {torch.cuda.get_device_name(0)}")

BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()


Device: cuda | Tesla T4


In [12]:
# Cell 4 — Load data, stratified split, class weights
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(raw)

def to_df(split):
    d = raw[split].to_pandas()
    d = d.rename(columns={"TWEET": "text", "LABEL": "raw_label"})
    d["stance"] = d["raw_label"].map(RAW_ID2CANON)
    d["label"] = d["stance"].map(LABEL2ID)
    return d[["text", "stance", "label"]].dropna().reset_index(drop=True)

train_pool = pd.concat([to_df("train"), to_df("validation")], ignore_index=True)
test_df    = to_df("test")   # held out, untouched until the final evaluation

# Stratified train/validation split carved out of the training pool only.
# The official "test" split is never used for model/rank selection.
train_df, val_df = train_test_split(
    train_pool,
    test_size=0.15,
    random_state=SEED,
    stratify=train_pool["label"],
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print("\nClass distribution (train / val / test):")
dist = pd.DataFrame({
    "train": train_df["stance"].value_counts(),
    "val":   val_df["stance"].value_counts(),
    "test":  test_df["stance"].value_counts(),
}).fillna(0).astype(int)
print(dist)
print(f"\nSizes -> train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Class weights for imbalance handling, computed on TRAIN ONLY
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(LABELS)),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights:", dict(zip(LABELS, class_weights.tolist())))


DatasetDict({
    train: Dataset({
        features: ['id_str', 'TWEET', 'LABEL'],
        num_rows: 6046
    })
    test: Dataset({
        features: ['id_str', 'TWEET', 'LABEL'],
        num_rows: 2016
    })
    validation: Dataset({
        features: ['id_str', 'TWEET', 'LABEL'],
        num_rows: 2015
    })
})

Class distribution (train / val / test):
         train  val  test
stance                   
AGAINST   2784  492   829
NONE      1265  223   380
SUPPORT   2802  495   807

Sizes -> train: 6851, val: 1210, test: 2016
Class weights: {'SUPPORT': 0.8150131106376648, 'AGAINST': 0.8202825784683228, 'NONE': 1.8052700757980347}


In [13]:
# Cell 5 — Tokenizer + Dataset + DataLoaders
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = StanceDataset(train_df, tokenizer, MAX_LEN)
val_ds   = StanceDataset(val_df, tokenizer, MAX_LEN)
test_ds  = StanceDataset(test_df, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [14]:
# Cell 6 — QLoRA model builder + evaluate()
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label={i: l for i, l in enumerate(LABELS)},
        label2id=LABEL2ID,
        quantization_config=BNB_CONFIG,
        torch_dtype=torch.bfloat16,     # dtype for the NON-quantized parts (e.g. the score head)
        attn_implementation="eager",
        device_map={"": 0},             # 4-bit models must be placed at load time, not via .to()
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False      # required alongside gradient checkpointing
    model = prepare_model_for_kbit_training(model)
    return model


@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        out = model(input_ids=input_ids, attention_mask=attn_mask)
        loss = loss_fn(out.logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = torch.argmax(out.logits, dim=-1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    per_class_f1 = f1_score(all_labels, all_preds, average=None,
                             labels=list(range(len(LABELS))))
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": acc,
        "macro_f1": macro_f1,
        "per_class_f1": dict(zip(LABELS, per_class_f1)),
        "preds": all_preds,
        "labels": all_labels,
    }


In [15]:
# Cell 7 — Training loop for one LoRA rank (QLoRA: quantized frozen base + fp32 LoRA adapters)
def run_experiment(rank):
    print(f"\n{'='*70}\nLoRA rank = {rank}\n{'='*70}")
    set_seed(SEED)  # identical init/data order across ranks -> fair comparison

    model = build_model()
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=rank,
        lora_alpha=rank * LORA_ALPHA_MULT,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        modules_to_save=["score"],   # keep the classification head trainable, full precision
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    loss_fn = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    ckpt_dir = os.path.join(OUT_DIR, f"ckpt_rank{rank}")
    best_macro_f1 = -1.0
    best_epoch = -1
    epochs_no_improve = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f"rank={rank} epoch={epoch}/{EPOCHS}")
        for batch in pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            out = model(input_ids=input_ids, attention_mask=attn_mask)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()

            running_loss += loss.item() * labels.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        train_loss = running_loss / len(train_loader.dataset)
        val_metrics = evaluate(model, val_loader, loss_fn)
        history.append({
            "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
        })
        print(f"  epoch {epoch}: train_loss={train_loss:.4f} "
              f"val_loss={val_metrics['loss']:.4f} "
              f"val_acc={val_metrics['accuracy']:.4f} "
              f"val_macro_f1={val_metrics['macro_f1']:.4f}")

        # Checkpointing: only keep the adapter from the best epoch so far
        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_val_metrics = val_metrics
            epochs_no_improve = 0
            model.save_pretrained(ckpt_dir)
        else:
            epochs_no_improve += 1

        # Early stopping
        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} "
                  f"(no val Macro-F1 improvement for {PATIENCE} epochs).")
            break

    result = {
        "rank": rank,
        "trainable_params": trainable_params,
        "best_epoch": best_epoch,
        "val_accuracy": best_val_metrics["accuracy"],
        "val_macro_f1": best_val_metrics["macro_f1"],
        "val_per_class_f1": best_val_metrics["per_class_f1"],
        "ckpt_dir": ckpt_dir,
        "history": history,
    }

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()
    return result


In [16]:
# Cell 8 — Train the selected rank (validation set only — test set is never touched here)
# RANKS has one entry (16); the loop is kept so you can drop more ranks back in later.
sweep_results = []
for r in RANKS:
    sweep_results.append(run_experiment(r))



LoRA rank = 16


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 8,800,896 || all params: 502,836,352 || trainable%: 1.7503


rank=16 epoch=1/6:   0%|          | 0/429 [00:00<?, ?it/s]

...

  epoch 4: train_loss=0.4667 val_loss=0.7612 val_acc=0.7388 val_macro_f1=0.7480


rank=16 epoch=5/6:   0%|          | 0/429 [00:00<?, ?it/s]

  epoch 5: train_loss=0.3758 val_loss=0.7856 val_acc=0.7405 val_macro_f1=0.7446


rank=16 epoch=6/6:   0%|          | 0/429 [00:00<?, ?it/s]

  epoch 6: train_loss=0.3472 val_loss=0.7485 val_acc=0.7545 val_macro_f1=0.7630


In [17]:
# Cell 9 — Result summary (single rank now, but same table shape as a multi-rank sweep)
comparison_df = pd.DataFrame([{
    "LoRA Rank": r["rank"],
    "Trainable Params": r["trainable_params"],
    "Best Epoch": r["best_epoch"],
    "Val Accuracy": round(r["val_accuracy"], 4),
    "Val Macro-F1": round(r["val_macro_f1"], 4),
    **{f"Val F1 ({k})": round(v, 4) for k, v in r["val_per_class_f1"].items()},
} for r in sweep_results]).sort_values("Val Macro-F1", ascending=False).reset_index(drop=True)

print("=" * 70)
print("LoRA RANK COMPARISON (selection metric: Validation Macro-F1)")
print("=" * 70)
print(comparison_df.to_string(index=False))
comparison_df


LoRA RANK COMPARISON (selection metric: Validation Macro-F1)
 LoRA Rank  Trainable Params  Best Epoch  Val Accuracy  Val Macro-F1  Val F1 (SUPPORT)  Val F1 (AGAINST)  Val F1 (NONE)
        16           8800896           6        0.7545         0.763            0.7465            0.7416         0.8009


,LoRA Rank,Trainable Params,Best Epoch,Val Accuracy,Val Macro-F1,Val F1 (SUPPORT),Val F1 (AGAINST),Val F1 (NONE)
0,16,8800896,6,0.7545,0.763,0.7465,0.7416,0.8009


In [18]:
# Cell 10 — Select best rank by validation Macro-F1, evaluate ONCE on test set
best_result = max(sweep_results, key=lambda r: r["val_macro_f1"])
best_rank = best_result["rank"]
print(f"Selected configuration: LoRA rank = {best_rank} "
      f"(val Macro-F1 = {best_result['val_macro_f1']:.4f})")

final_base = build_model()
final_model = PeftModel.from_pretrained(final_base, best_result["ckpt_dir"])

test_loss_fn = nn.CrossEntropyLoss(weight=class_weights)
test_metrics = evaluate(final_model, test_loader, test_loss_fn)

print("\n" + "=" * 70)
print(f"FINAL TEST-SET RESULTS (LoRA rank {best_rank}, evaluated once)")
print("=" * 70)
print(f"Test Accuracy : {test_metrics['accuracy']:.4f}")
print(f"Test Macro-F1 : {test_metrics['macro_f1']:.4f}")
print("\nPer-class F1:")
for name, f1v in test_metrics["per_class_f1"].items():
    print(f"  {name:8s}: {f1v:.4f}")

print("\nFull classification report:")
print(classification_report(
    test_metrics["labels"], test_metrics["preds"],
    target_names=LABELS, digits=4,
))


Selected configuration: LoRA rank = 16 (val Macro-F1 = 0.7630)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



FINAL TEST-SET RESULTS (LoRA rank 16, evaluated once)
Test Accuracy : 0.7654
Test Macro-F1 : 0.7814

Per-class F1:
  SUPPORT : 0.7385
  AGAINST : 0.7511
  NONE    : 0.8545

Full classification report:
              precision    recall  f1-score   support

     SUPPORT     0.7300    0.7472    0.7385       807
     AGAINST     0.7580    0.7443    0.7511       829
        NONE     0.8590    0.8500    0.8545       380

    accuracy                         0.7654      2016
   macro avg     0.7824    0.7805    0.7814      2016
weighted avg     0.7658    0.7654    0.7655      2016



In [19]:
# Cell 11 — Save the best adapter + final summary table
final_adapter_dir = os.path.join(OUT_DIR, "best_lora_adapter")
final_model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Best adapter (rank={best_rank}) saved to: {final_adapter_dir}")

test_summary_df = pd.DataFrame([{
    "LoRA Rank": best_rank,
    "Test Accuracy": round(test_metrics["accuracy"], 4),
    "Test Macro-F1": round(test_metrics["macro_f1"], 4),
    **{f"Test F1 ({k})": round(v, 4) for k, v in test_metrics["per_class_f1"].items()},
}])
print("\nFinal test summary table:")
print(test_summary_df.to_string(index=False))
test_summary_df


Best adapter (rank=16) saved to: /kaggle/working/best_lora_adapter

Final test summary table:
 LoRA Rank  Test Accuracy  Test Macro-F1  Test F1 (SUPPORT)  Test F1 (AGAINST)  Test F1 (NONE)
        16         0.7654         0.7814             0.7385             0.7511          0.8545


,LoRA Rank,Test Accuracy,Test Macro-F1,Test F1 (SUPPORT),Test F1 (AGAINST),Test F1 (NONE)
0,16,0.7654,0.7814,0.7385,0.7511,0.8545
